In [1]:
import copy
import io
import numpy as np
import pandas as pd
import spacy
import sys
import warnings

from contextlib import redirect_stdout
from dowhy import CausalModel
from econml.inference import BootstrapInference
from importlib import import_module
from pathlib import Path
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LassoCV
from sklearn.ensemble import GradientBoostingRegressor

module_path = str(Path("..").resolve())
if module_path not in sys.path:
    sys.path.append(module_path)

import utils.data_io as data_io
import utils.graph_utils as graph_utils

from config_handlers.causal_handler import CausalHandler
from config_handlers.rk_handler import RKHandler

warnings.filterwarnings("ignore")

/opt/homebrew/Caskroom/miniforge/base/envs/causal/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Utils

In [2]:
def make_nx_graphs(resps):
    graphs = {}
    for r in resps:
        print(f"Processing sample id: {r['question_id']}")
        graphs[str(r["question_id"])] = graph_utils.rk_to_nx(
            r["triple_objs"], return_full=True
        )
        print("-" * 40)
    return graphs

In [3]:
# Credit: https://github.com/BramVanroy/spacy_download
def load_spacy(model_name: str, **kwargs) -> spacy.Language:
    """Load a spaCy model, download it if it has not been installed yet.
    :param model_name: the model name, e.g., en_core_web_sm
    :param kwargs: options passed to the spaCy loader, such as component exclusion, as you
    would with spacy.load()
    :return: an initialized spaCy Language
    :raises: SystemExit: if the model_name cannot be downloaded
    """
    try:
        model_module = import_module(model_name)
    except ModuleNotFoundError:
        spacy.cli.download(model_name)
        model_module = import_module(model_name)

    return model_module.load(**kwargs)

In [4]:
def get_nouns(doc: spacy.tokens.Doc, return_dict: bool = True) -> dict | list:
    accepted_pos_tags = ["NN", "NNP", "NNPS", "NNS"]
    """ If needed, add pronouns: ["PRP", "PRP$", "WP", "WP$"] """
    accepted_dep_tags = [
        "nsubj",
        "nsubjpass",
        "dobj",
        "attr",
        "agent",
        "oprd",
        "amod",
        "nummod",
        "compound",
        "nmod",
        "pobj",
        "conj",
    ]

    resp_concepts = []
    for token in doc:
        if token.tag_ in accepted_pos_tags:
            # Return lemma_ insted of text
            if return_dict:
                new_entry = {
                    "text": token.lemma_,
                    "tag": token.tag_,
                    "subtree": [t.text for t in list(token.subtree)],
                }
            else:
                new_entry = token.lemma_
            resp_concepts.append(new_entry)
    return resp_concepts

In [5]:
def run_causal_inference_DML(
    data,
    nx_graph,
    out_var,
    test_significance=False,
    confidence_intervals=False,
    get_stderr=False,
    do_refute=False,
):
    all_estimates = {}
    all_refutations = {}

    # Prepare data for multinomial logistic regressor
    # responses = data["y"]  # 1-d vector of the responses
    cols_to_skip = ["question_id", "size", "response"]
    cols_to_keep = []
    for col in list(set(data.keys()) - set(cols_to_skip)):
        if col.startswith("y_") and col != out_var:
            continue
        cols_to_keep.append(col)
    data_for_estimation = data[cols_to_keep]  # n_observations x regressors
    regressors = list(data_for_estimation.drop(out_var, axis=1).keys())

    for col in regressors:
        print(f"Working on {col}...")

        # Exclude the current 'col' from the variables that can condition the estimates
        effect_modifiers = [r for r in regressors if r != col]

        # effect_modifiers: estimates are separately computed (conditioned)
        # for each value of the effect_modifier.
        model = CausalModel(
            data=data_for_estimation,
            treatment=col,
            outcome=out_var,
            graph=nx_graph,
            effect_modifiers=effect_modifiers,
        )
        estimand = model.identify_effect(proceed_when_unidentifiable=True)

        # Define method params
        # If less than default n_splits (5), use only 1 split
        n_split = 2 if len(data) < 5 else 5
        method_params = {
            "init_params": {
                "model_y": GradientBoostingRegressor(),
                "model_t": GradientBoostingRegressor(),
                "model_final": LassoCV(fit_intercept=False, cv=n_split),
                "featurizer": PolynomialFeatures(degree=1, include_bias=False),
            },
            "fit_params": {},
        }
        if test_significance:
            method_params["fit_params"] = {
                "inference": BootstrapInference(n_bootstrap_samples=1, n_jobs=-1)
            }

        # Control and treatment values are correct since we are occluding (removing)
        # elements from images.
        try:
            ce_estimate = model.estimate_effect(
                estimand,
                method_name="backdoor.econml.dml.DML",
                control_value=1,
                treatment_value=0,
                fit_estimator=True,
                method_params=method_params,
            )
        except Exception as e:
            print("Unable to run estimation. Got the following exception:")
            print(e)
            ce_estimate = None
            all_estimates[col] = ce_estimate
            continue

        # Run refutation with Placebo method
        if do_refute:
            print("- Running refutation test for estimate...")
            try:
                all_refutations[col] = model.refute_estimate(
                    estimand,
                    ce_estimate,
                    method_name="placebo_treatment_refuter",
                    show_progress_bar=False,
                    placebo_type="permute",
                )
            except Exception as e:
                print(
                    "Unable to run refutation test of obtained estimate. Got the following exception:"
                )
                print(e)
                all_refutations[col] = None

        # The following calls are is wrapped in a try-except because it might fail
        # when test_significance, confidence_intervals, or get_stderr are True.
        # This is due to incomplete integration between the DoWhy and econML libraries.

        if test_significance:
            print("- Running significance test for estimate...")
            try:
                ce_estimate.test_stat_significance()
            except Exception as e:
                print(
                    "Unable to test significance of obtained estimate. Got the following exception:"
                )
                print(e)

        if confidence_intervals:
            print("- Getting confidence intervals for estimate...")
            try:
                ce_estimate.get_confidence_intervals()
            except Exception as e:
                print(
                    "Unable to get confidence intervals of the obtained estimate. Got the following exception:"
                )
                print(e)

        # Compute standard error for the estimate
        if get_stderr:
            print("- Getting standard error for estimate...")
            try:
                ce_estimate.get_standard_error()
            except Exception as e:
                print(
                    "Unable to get the standard error for the obtained estimate. Got the following exception:"
                )
                print(e)

        all_estimates[col] = ce_estimate

    return all_estimates, all_refutations

In [6]:
def format_estimates(
    estimates,
    refutations=None,
    test_significance=False,
    confidence_intervals=False,
    std_error=False,
):
    out_data = {}

    for var_name, curr_est in estimates.items():
        # var_name = curr_est._treatment_name[0]
        if var_name not in out_data:
            out_data[var_name] = {}

        if curr_est is None:
            continue

        out_data[var_name] = {
            "estimate": curr_est.value.item(),
            "estimand_type": curr_est.target_estimand.estimand_type.value,
            "estimand_expr": curr_est.realized_estimand_expr,
        }

        # Catch the output of the interpret() method
        f = io.StringIO()
        with redirect_stdout(f):
            curr_est.interpret()
        out_data[var_name]["interpretation"] = f.getvalue().strip()

        if test_significance:
            try:
                curr_est.test_stat_significance()
                p_val = curr_est.test_stat_significance()["p_value"]
                if isinstance(p_val, tuple):
                    out_data[var_name]["p_value"] = list(p_val)
                elif isinstance(p_val, np.floating):
                    out_data[var_name]["p_value"] = p_val.item()
            except Exception as _:
                out_data[var_name]["p_value"] = None

        if std_error:
            try:
                out_data[var_name]["std_error"] = curr_est.get_standard_error()
            except Exception as _:
                out_data[var_name]["std_error"] = None

        if confidence_intervals:
            try:
                out_data[var_name]["conf_intervals"] = (
                    curr_est.get_confidence_intervals()
                )
            except Exception as _:
                out_data[var_name]["conf_intervals"] = None

        if refutations:
            refute_data = refutations[var_name]
            if refute_data:
                p_val = refute_data.refutation_result["p_value"]
                out_data[var_name]["refute_test"] = {
                    "is_stat_significant": refute_data.refutation_result[
                        "is_statistically_significant"
                    ].item(),
                }
                if isinstance(p_val, tuple):
                    out_data[var_name]["refute_test"]["p_value"] = list(p_val)
                elif isinstance(p_val, np.floating):
                    out_data[var_name]["refute_test"]["p_value"] = p_val.item()
            else:
                out_data[var_name]["refute_test"] = {}
        else:
            out_data[var_name]["refute_test"] = {}

    return out_data

# Main

In [7]:
MODEL = "sharegpt4v"
DATASET = "llava-bench"

In [8]:
TEST_SIGNIFICANCE = False
CONF_INTERVALS = False
OUTPUT_STDERR = False
DO_REFUTE = False

if TEST_SIGNIFICANCE or CONF_INTERVALS or OUTPUT_STDERR:
    print(
        "WARNING: The integration of DoWhy and econML is incomplete. "
        "Computations for testing significance, retrieving confidence "
        "intervals, and standard error for the obtained estimate might fail."
    )

In [9]:
# Load data handlers
rk_hdl = RKHandler()
rk_hdl.set_curr_model(MODEL)
rk_hdl.set_curr_ds(DATASET)
causal_hdl = CausalHandler()
causal_hdl.set_curr_ds(DATASET)
causal_hdl.set_curr_model(MODEL)

In [10]:
base_dir = Path("..")
resps = data_io.load_jsonl(base_dir.joinpath(rk_hdl.get_rk_final_path()))
counter_resps = data_io.load_jsonl(
    base_dir.joinpath(causal_hdl.get_counter_resps_path())
)

In [11]:
print("=" * 60)
print(f"Processing data for {MODEL} + {DATASET}...")
""" Preparing data """
resps_ids = set([q["question_id"] for q in resps])
dict_counter_resp = {
    q_idx: next(r["response"] for r in counter_resps if r["question_id"] == q_idx)
    for q_idx in resps_ids
}

# Merge initial and counterfactual responses
all_resps = copy.deepcopy(counter_resps)
for q_idx in resps_ids:
    og_resp = next((r for r in resps if r["question_id"] == q_idx), None)
    if og_resp:
        all_resps.append(
            {
                "question_id": str(og_resp["question_id"]),
                "size": 0,
                "occluded": [],
                "response": og_resp["response"],
            }
        )
    else:
        print(f"We have an issue with question {q_idx}.")

Processing data for sharegpt4v + llava-bench...


In [12]:
df = pd.DataFrame(all_resps)
df.drop(["path"], axis=1, inplace=True)
# Force question_id column type to string
df["question_id"] = df["question_id"].map(str)
df["response"] = df["response"].apply(lambda x: x.lower())
df.head()

,question_id,occluded,size,response
0,0,[new-diamond head crater],1,the famous sight in the photo is the diamond h...
1,0,[new-honolulu],1,the famous sight in the photo is the diamond h...
2,0,[new-beach and body of water],1,the famous sight in the photo is the diamond h...
3,0,"[new-diamond head crater, new-honolulu]",2,the famous sight in the photo is the diamond h...
4,0,"[new-diamond head crater, new-beach and body o...",2,the famous sight in the photo is the hawaii vo...


In [13]:
max_triplets = {
    str(q_idx): max([int(q["size"]) for q in all_resps if q["question_id"] == q_idx])
    for q_idx in resps_ids
}

# This is done to extract an actual list of concepts, not any other PD objects.
dict_concepts = {
    str(q_idx): df[(df["question_id"] == q_idx) & (df["size"] == max_triplets[q_idx])][
        "occluded"
    ].iloc[0]
    for q_idx in max_triplets
}

In [ ]:
# Make dictionary of dataframes w.r.t. question id
out_f = Path("data_oe")
out_f.mkdir(parents=True, exist_ok=True)
file_name = Path(out_f, f"df_dict_{MODEL}_{DATASET}.pkl")  # Creating this for later
df_dict = {
    str(q_idx): df[df["question_id"] == str(q_idx)] for q_idx in dict_counter_resp
}

In [15]:
# Taking the list of occluded concepts and splitting it
for q_idx in df_dict:
    for concept in dict_concepts[q_idx]:
        df_dict[q_idx][concept] = 1
for q_idx in df_dict:
    for id, row in df_dict[q_idx].iterrows():
        curr_concepts = row["occluded"]
        for c in curr_concepts:
            df_dict[q_idx].loc[id, c] = 0
    df_dict[q_idx].drop(["occluded"], axis=1, inplace=True)

Extract concepts from responses with POS + dependency parsing

In [16]:
spacy_model_name = "en_core_web_trf"
print(f"Loading spaCy model: {spacy_model_name}...")
to_exclude = ["ner", "parser"]  # do not load NER and dependency parsing modules
nlp = load_spacy(spacy_model_name, exclude=to_exclude)
print(nlp.pipe_names)

Loading spaCy model: en_core_web_trf...
['transformer', 'tagger', 'attribute_ruler', 'lemmatizer']


In [17]:
"""OLD: adds every single 'new' concept as an outcome variable"""

# # Prepare columns with nouns extracted from the responses
# unique_nouns = {}
# for q_idx, q_df in df_dict.items():
#     # Parse all responses
#     q_nouns = {}
#     for idx in q_df.index.tolist():
#         q_nouns[idx] = get_nouns(nlp(q_df.loc[idx]["response"]), return_dict=False)

#     # Merge all list of nouns are reduce to a single set
#     unique = set(elem for curr in q_nouns.values() for elem in curr)
#     # Remove from the unique nouns the ones that are mentioned in the RKs.
#     # Otherwise, a concept (variable) is used to estimate itself (i.e., as outcome)
#     clean_concepts = ["-".join(e.split("-")[1:]) for e in dict_concepts[q_idx]]
#     lemma_concepts = set([token.lemma_ for cc in clean_concepts for token in nlp(cc)])
#     unique_nouns[q_idx] = list(unique.difference(lemma_concepts))

#     # Add binary columns for each noun found in the response. Initialise with 1s.
#     for noun in unique_nouns[q_idx]:
#         q_df[f"y_{noun}"] = 1
#     for id, row in q_df.iterrows():
#         for noun in unique_nouns[q_idx]:
#             if noun not in q_nouns[id]:
#                 q_df.loc[id, f"y_{noun}"] = 0
#     print(f"Handled question w/ id: {q_idx} -> {len(q_df)} rows")

"OLD: adds every single 'new' concept as an outcome variable"

In [17]:
"""Only use the nouns that are present in the original response to do estimation."""

# Prepare columns with nouns extracted from the responses
unique_counter = {}
unique_og = {}
for q_idx, q_df in df_dict.items():
    # Parse original response (i.e., the non-counterfactual one) separately
    unique_counter[q_idx] = {}
    for idx in q_df.index.tolist():
        size = q_df.loc[idx]["size"]
        # These are lemmas already
        nouns = get_nouns(nlp(q_df.loc[idx]["response"]), return_dict=False)
        unique_counter[q_idx][idx] = list(set(nouns))
        if size == 0:
            unique_og[q_idx] = list(set(nouns))

    # Merge all lists of nouns from counterfatuals to a single set
    # unique_counter[q_idx] = list(
    #     set(elem for curr in q_nouns.values() for elem in curr)
    # )
    print(f"Handled question: {q_idx}.")

Handled question: 0.
Handled question: 1.
Handled question: 2.
Handled question: 4.
Handled question: 6.
Handled question: 7.
Handled question: 8.
Handled question: 9.
Handled question: 10.
Handled question: 11.
Handled question: 12.
Handled question: 13.
Handled question: 14.
Handled question: 16.
Handled question: 19.
Handled question: 20.
Handled question: 21.
Handled question: 22.
Handled question: 23.
Handled question: 24.
Handled question: 25.
Handled question: 26.
Handled question: 27.
Handled question: 28.
Handled question: 29.
Handled question: 30.
Handled question: 31.
Handled question: 32.
Handled question: 33.
Handled question: 34.
Handled question: 35.
Handled question: 36.
Handled question: 37.
Handled question: 38.
Handled question: 39.
Handled question: 40.
Handled question: 42.
Handled question: 43.
Handled question: 44.
Handled question: 45.
Handled question: 47.
Handled question: 49.
Handled question: 50.
Handled question: 52.
Handled question: 53.
Handled question: 

In [18]:
# Add binary columns for each noun found in the OG response.
for q_idx, q_df in df_dict.items():
    # Initialise with 1s.
    for noun in unique_og[q_idx]:
        q_df[f"y_{noun}"] = 1

In [19]:
def print_cols(df_dict):
    for q_idx in df_dict.keys():
        print(f">> Working with question # {q_idx} -- {len(df_dict[q_idx])} rows.")
        curr_df = df_dict[q_idx]
        # For each y_x column, check that we have both 0s and 1s.
        skip = ["question_id", "size", "response"]
        cols = list(filter(lambda x: x not in skip, curr_df.columns))
        regressors = [col for col in cols if not col.startswith("y_")]
        outcomes = [col for col in cols if col.startswith("y_")]
        print("Regressors:")
        for r in regressors:
            print(f" - {r}: {curr_df[r].unique()}")
        print("Outcomes:")
        for o in outcomes:
            print(f" - {o}: {curr_df[o].unique()}")
        break

In [20]:
print_cols(df_dict)

>> Working with question # 0 -- 8 rows.
Regressors:
 - new-diamond head crater: [0 1]
 - new-honolulu: [1 0]
 - new-beach and body of water: [1 0]
Outcomes:
 - y_crater: [1]
 - y_hawaii: [1]
 - y_diamond: [1]
 - y_photo: [1]
 - y_head: [1]
 - y_honolulu: [1]
 - y_sight: [1]


In [21]:
for q_idx, q_df in df_dict.items():
    # Iterate over the rows and if a column is not included in the list
    for row_id in q_df.index:
        for noun in unique_og[q_idx]:
            if noun not in unique_counter[q_idx][row_id]:
                q_df.loc[row_id, f"y_{noun}"] = 0
    print(f"Handled question: {q_idx} -> {q_df.shape[0]} rows x {q_df.shape[1]} cols.")

Handled question: 0 -> 8 rows x 13 cols.
Handled question: 1 -> 128 rows x 49 cols.
Handled question: 2 -> 256 rows x 48 cols.
Handled question: 4 -> 4 rows x 8 cols.
Handled question: 6 -> 256 rows x 39 cols.
Handled question: 7 -> 858 rows x 29 cols.
Handled question: 8 -> 128 rows x 51 cols.
Handled question: 9 -> 64 rows x 36 cols.
Handled question: 10 -> 1232 rows x 49 cols.
Handled question: 11 -> 4 rows x 9 cols.
Handled question: 12 -> 16 rows x 20 cols.
Handled question: 13 -> 4 rows x 30 cols.
Handled question: 14 -> 256 rows x 48 cols.
Handled question: 16 -> 1232 rows x 27 cols.
Handled question: 19 -> 16 rows x 36 cols.
Handled question: 20 -> 460 rows x 24 cols.
Handled question: 21 -> 8 rows x 12 cols.
Handled question: 22 -> 256 rows x 31 cols.
Handled question: 23 -> 612 rows x 44 cols.
Handled question: 24 -> 734 rows x 42 cols.
Handled question: 25 -> 256 rows x 31 cols.
Handled question: 26 -> 64 rows x 23 cols.
Handled question: 27 -> 32 rows x 12 cols.
Handled que

In [22]:
print_cols(df_dict)

>> Working with question # 0 -- 8 rows.
Regressors:
 - new-diamond head crater: [0 1]
 - new-honolulu: [1 0]
 - new-beach and body of water: [1 0]
Outcomes:
 - y_crater: [1 0]
 - y_hawaii: [1]
 - y_diamond: [1 0]
 - y_photo: [1]
 - y_head: [1 0]
 - y_honolulu: [1 0]
 - y_sight: [1]


In [23]:
# temp -- remove later; or add loading above and skip processing
pd.to_pickle(df_dict, file_name)
# if not file_name.exists():
#     pd.to_pickle(df_dict, file_name)
# else:
#     df_dict = pd.read_pickle(file_name)

In [24]:
for q_idx, q_df in df_dict.items():
    print(f"{q_idx}: {q_df.shape[0]} rows x {q_df.shape[1]} columns.")
    n_input_vars = len(dict_concepts[q_idx])
    print(f" - Variables: {n_input_vars}")
    n_outcome_vars = len([col for col in q_df.columns if col.startswith("y_")])
    print(f" - Outcomes: {n_outcome_vars}")
    # Remove RKs so that we do not estimate one concept with itself
    # n_reduced_vars = n_outcome_vars - len(dict_concepts[q_idx])
    # print(f" - Potential reduction: {n_reduced_vars}")

0: 8 rows x 13 columns.
 - Variables: 3
 - Outcomes: 7
1: 128 rows x 49 columns.
 - Variables: 7
 - Outcomes: 39
2: 256 rows x 48 columns.
 - Variables: 8
 - Outcomes: 37
4: 4 rows x 8 columns.
 - Variables: 2
 - Outcomes: 3
6: 256 rows x 39 columns.
 - Variables: 8
 - Outcomes: 28
7: 858 rows x 29 columns.
 - Variables: 12
 - Outcomes: 14
8: 128 rows x 51 columns.
 - Variables: 7
 - Outcomes: 41
9: 64 rows x 36 columns.
 - Variables: 6
 - Outcomes: 27
10: 1232 rows x 49 columns.
 - Variables: 15
 - Outcomes: 31
11: 4 rows x 9 columns.
 - Variables: 2
 - Outcomes: 4
12: 16 rows x 20 columns.
 - Variables: 4
 - Outcomes: 13
13: 4 rows x 30 columns.
 - Variables: 2
 - Outcomes: 25
14: 256 rows x 48 columns.
 - Variables: 8
 - Outcomes: 37
16: 1232 rows x 27 columns.
 - Variables: 15
 - Outcomes: 9
19: 16 rows x 36 columns.
 - Variables: 4
 - Outcomes: 29
20: 460 rows x 24 columns.
 - Variables: 9
 - Outcomes: 12
21: 8 rows x 12 columns.
 - Variables: 3
 - Outcomes: 6
22: 256 rows x 31 co

In [ ]:
# # Rename outcome columns as y_<concept_noun>
# # Code above updated, no need to run this cell again in the future.
# for q_idx, q_df in df_dict.items():
#     to_ignore = ["question_id", "size", "response"]
#     to_ignore.extend(dict_concepts[q_idx])
#     for col in q_df.columns:
#         if col not in to_ignore:
#             q_df.rename(columns={col: f"y_{col}"}, inplace=True)

In [ ]:
# # Temporarily remove y_x columns and re-add them later
# for q_idx, q_df in df_dict.items():
#     cols = [col for col in q_df.columns if col.startswith("y_")]
#     try:
#         df_dict[q_idx] = q_df.drop(cols, axis=1, inplace=False)
#     except Exception as _:
#         print(f"Columns {cols} not present.")

Prepare data for causal inference

In [25]:
# First, we create the NX graphs based on the data contained from 'resps'.
# This is done so that we can create the '.dot' string required by 'dowhy'.
rk_graphs = make_nx_graphs(resps)
img_ids = list(rk_graphs.keys())
print(f"Loaded {len(rk_graphs)} samples.")
print(f"Duplicate ids: {set([x for x in img_ids if img_ids.count(x) > 1])}")

Processing sample id: 0
- No self-loops found.
- No cycles found.
----------------------------------------
Processing sample id: 1
- No self-loops found.
- Identified cycles: [[['new-diamond head crater', '14430-city'], ['14430-city', 'new-diamond head crater']], [['new-urban life', '14430-city'], ['14430-city', 'new-urban life']]]
----------------------------------------
Processing sample id: 2
- No self-loops found.
- No cycles found.
----------------------------------------
Processing sample id: 4
- No self-loops found.
- No cycles found.
----------------------------------------
Processing sample id: 6
- No self-loops found.
- No cycles found.
----------------------------------------
Processing sample id: 7
- No self-loops found.
- No cycles found.
----------------------------------------
Processing sample id: 8
- No self-loops found.
- No cycles found.
----------------------------------------
Processing sample id: 9
- No self-loops found.
- Identified cycles: [[['19592-dog', 'new-m

In [26]:
# Add to each graph a node corresponding to the outcome variable 'y'.
# Here, since we have multiple variables, we need to add several of these nodes.
# Can't add all of the y_i nodes to the graph because then the estimators don't know
# what to do. Need to make separate graphs, each with a y_i.

exp_rk_graphs = {}
for rk_idx, rk_g in rk_graphs.items():
    exp_rk_graphs[rk_idx] = {}
    for noun in unique_og[rk_idx]:
        exp_rk_graphs[rk_idx][f"y_{noun}"] = copy.deepcopy(rk_g)
        exp_rk_graphs[rk_idx][f"y_{noun}"].add_node(f"y_{noun}", label=f"y_{noun}")

        for node in list(exp_rk_graphs[rk_idx][f"y_{noun}"].nodes()):
            if node == f"y_{noun}":
                continue
            exp_rk_graphs[rk_idx][f"y_{noun}"].add_edge(node, f"y_{noun}")

In [27]:
# Run causal analysis
causal_out_path = base_dir.joinpath(causal_hdl.get_estimates_path()).resolve()
data_io.make_dir(causal_out_path.parent)
print(f"Saving estimates to: {str(causal_out_path)}")

Saving estimates to: /Users/lorenzocorti/Developer/mllm_diagnosis/data/causal_effects/estimates/sharegpt4v/llava-bench/ce.jsonl


In [28]:
# Load data currently available
if causal_out_path.exists():
    old_data = data_io.load_jsonl(causal_out_path)
    old_ids = set([od["question_id"] for od in old_data])
else:
    print("First run.")
    old_ids = []

First run.


Run estimation

In [29]:
for q_idx, rk_data in exp_rk_graphs.items():
    if q_idx in old_ids:
        print(f"Already processed id {q_idx}. Moving to next one.")
        continue

    print("=" * 60)
    print(f">> Working with question # {q_idx} -- Samples: {len(df_dict[q_idx])}")
    curr_df = df_dict[q_idx]

    outcome_vars = [v for v in df_dict[q_idx].columns if v.startswith("y_")]
    out_data = {"question_id": q_idx, "estimates": {}}
    for out_var in outcome_vars:
        curr_rk = rk_data[out_var].copy()
        print(f"Current outcome var: {out_var}")
        # Compute causal effects
        estimates, refutations = run_causal_inference_DML(
            curr_df,
            curr_rk,
            out_var,
            test_significance=TEST_SIGNIFICANCE,
            confidence_intervals=CONF_INTERVALS,
            get_stderr=OUTPUT_STDERR,
            do_refute=DO_REFUTE,
        )

        # Format and save results of causal analysis
        out_data["estimates"][out_var] = format_estimates(
            estimates,
            refutations=refutations,
            test_significance=TEST_SIGNIFICANCE,
            confidence_intervals=CONF_INTERVALS,
            std_error=OUTPUT_STDERR,
        )
    # Still save data after every sample is done
    data_io.append_to_jsonl([out_data], causal_out_path)

print("=" * 30)
print(f"Done for {MODEL} x {DATASET}.")
print("=" * 30)

>> Working with question # 0 -- Samples: 8
Current outcome var: y_crater
Working on new-diamond head crater...
Working on new-honolulu...
Working on new-beach and body of water...
Current outcome var: y_hawaii
Working on new-diamond head crater...
Working on new-honolulu...
Working on new-beach and body of water...
Current outcome var: y_diamond
Working on new-diamond head crater...
Working on new-honolulu...
Working on new-beach and body of water...
Current outcome var: y_photo
Working on new-diamond head crater...
Working on new-honolulu...
Working on new-beach and body of water...
Current outcome var: y_head
Working on new-diamond head crater...
Working on new-honolulu...
Working on new-beach and body of water...
Current outcome var: y_honolulu
Working on new-diamond head crater...
Working on new-honolulu...
Working on new-beach and body of water...
Current outcome var: y_sight
Working on new-diamond head crater...
Working on new-honolulu...
Working on new-beach and body of water...

**[Old] Exploring spaCy**

In [ ]:
# sample_text = df.iloc[0]["response"]
# sample_doc = nlp(sample_text)
# print(sample_doc)
# svg = displacy.render(sample_doc, style="dep", jupyter=False)
# output_path = Path("sample_doc.svg")
# output_path.open("w", encoding="utf-8").write(svg)

In [ ]:
# Get a sense of the dependency labels available
pos_expl = {pos: spacy.explain(pos) for pos in nlp._meta["labels"]["tagger"]}
dep_expl = {dep: spacy.explain(dep) for dep in nlp._meta["labels"]["parser"]}